# Adquisición y normalización de datos de ventas

**Objetivo:** integrar transacciones (CSV) con el maestro de productos (Excel), limpiar los datos, calcular `total_venta` y exportar a Parquet.

## 1. Librerías

In [1]:
from pathlib import Path
import pandas as pd

RAW = Path('../data/raw')
OUT = Path('../data/processed/ventas_final.parquet')

## 2. Adquisición
### 2.1 Transacciones (CSV)

In [2]:
trx = pd.read_csv(RAW / 'transacciones.csv')
print(trx.shape)
trx.info()
trx.head()

(300, 4)
<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id_transaccion  300 non-null    int64  
 1   id_producto     295 non-null    float64
 2   cantidad        288 non-null    float64
 3   fecha           300 non-null    str    
dtypes: float64(2), int64(1), str(1)
memory usage: 12.4 KB


,id_transaccion,id_producto,cantidad,fecha
0,1,1.0,3.0,2026-04-21
1,2,8.0,NaN,2026-05-21
2,3,7.0,1.0,2026-05-14
3,4,5.0,7.0,2026-01-25
4,5,5.0,4.0,2026-05-29


### 2.2 Maestro de productos (Excel)
Se usa `engine='openpyxl'` y se lee `id_producto` como texto para ver el formato original.

In [3]:
prod = pd.read_excel(RAW / 'productos.xlsx', sheet_name='maestro', engine='openpyxl', dtype={'id_producto': str})
print(prod.shape)
prod

(11, 3)


,id_producto,nombre_producto,precio_unitario
0,001,Yerba mate 1kg,289.0
1,002,Dulce de leche 450g,145.5
2,003,Café molido 500g,410.0
3,004,Galletitas maría,62.0
4,005,Aceite girasol 1.5L,198.9
5,006,Arroz 1kg,79.0
6,007,Fideos 500g,55.5
7,008,Queso dambo kg,520.0
8,009,Leche entera 1L,42.0
9,010,Pan de molde,115.0


**Diagnóstico inicial:**
- `fecha` llega como texto (`object`) y tiene valores inválidos (`'sin fecha'`).
- `id_producto` es `float` en el CSV (por los nulos) y texto con ceros (`'001'`) en el Excel → tipos distintos, el merge fallaría.
- Hay nulos en `id_producto` y `cantidad`.
- El Excel tiene un producto duplicado.

In [4]:
print('Nulos en transacciones:')
print(trx.isna().sum())
print('\nValores de fecha no parseables:', pd.to_datetime(trx['fecha'], errors='coerce').isna().sum())
print('Duplicados en maestro:', prod.duplicated('id_producto').sum())

Nulos en transacciones:
id_transaccion     0
id_producto        5
cantidad          12
fecha              0
dtype: int64

Valores de fecha no parseables: 4
Duplicados en maestro: 1


## 3. Transformación
### 3.1 Tipos de datos y nulos en transacciones
- `fecha` → `datetime` (los inválidos quedan como `NaT`).
- Se **eliminan** filas sin `id_producto` o sin `fecha`: no se pueden atribuir a un producto ni a un período.
- `cantidad` nula se **imputa** con la mediana de ese producto (conserva la venta sin distorsionar el volumen).

In [5]:
trx['fecha'] = pd.to_datetime(trx['fecha'], format='%Y-%m-%d', errors='coerce')
antes = len(trx)
trx = trx.dropna(subset=['id_producto', 'fecha'])
print(f'Filas eliminadas: {antes - len(trx)}')

trx['cantidad'] = trx['cantidad'].fillna(trx.groupby('id_producto')['cantidad'].transform('median'))
trx['id_producto'] = trx['id_producto'].astype('int64')
trx['cantidad'] = trx['cantidad'].astype('int64')
trx.dtypes

Filas eliminadas: 9


id_transaccion             int64
id_producto                int64
cantidad                   int64
fecha             datetime64[us]
dtype: object

### 3.2 Maestro de productos
Se convierte `id_producto` de texto (`'001'`) a entero para que coincida con el CSV y se eliminan duplicados.

In [6]:
prod['id_producto'] = prod['id_producto'].str.strip().astype('int64')
prod['nombre_producto'] = prod['nombre_producto'].str.strip()
prod = prod.drop_duplicates(subset='id_producto')
prod.dtypes

id_producto          int64
nombre_producto        str
precio_unitario    float64
dtype: object

### 3.3 Merge por `id_producto`
`validate='many_to_one'` asegura que cada producto aparezca una sola vez en el maestro. `indicator=True` permite detectar transacciones con productos que no existen en el catálogo.

In [7]:
ventas = trx.merge(prod, on='id_producto', how='left', validate='many_to_one', indicator=True)
print(ventas['_merge'].value_counts())
ventas[ventas['_merge'] == 'left_only']

_merge
both          289
left_only       2
right_only      0
Name: count, dtype: int64


,id_transaccion,id_producto,cantidad,fecha,nombre_producto,precio_unitario,_merge
10,11,99,5,2026-05-31,NaN,NaN,left_only
20,21,99,3,2026-03-20,NaN,NaN,left_only


Las transacciones con `id_producto = 99` no existen en el maestro (no tienen nombre ni precio), por lo que se eliminan.

In [8]:
ventas = ventas[ventas['_merge'] == 'both'].drop(columns='_merge')

## 4. Normalización: columna calculada `total_venta`

In [9]:
ventas['total_venta'] = (ventas['cantidad'] * ventas['precio_unitario']).round(2)
ventas = ventas.sort_values(['fecha', 'id_transaccion']).reset_index(drop=True)
ventas.head(10)

,id_transaccion,id_producto,cantidad,fecha,nombre_producto,precio_unitario,total_venta
0,173,8,4,2026-01-01,Queso dambo kg,520.0,2080.0
1,245,7,6,2026-01-01,Fideos 500g,55.5,333.0
2,246,1,1,2026-01-01,Yerba mate 1kg,289.0,289.0
3,107,8,4,2026-01-02,Queso dambo kg,520.0,2080.0
4,190,4,5,2026-01-03,Galletitas maría,62.0,310.0
5,201,4,1,2026-01-03,Galletitas maría,62.0,62.0
6,239,4,4,2026-01-03,Galletitas maría,62.0,248.0
7,289,6,6,2026-01-03,Arroz 1kg,79.0,474.0
8,221,2,3,2026-01-05,Dulce de leche 450g,145.5,436.5
9,187,2,3,2026-01-06,Dulce de leche 450g,145.5,436.5


### Validación final: sin nulos en campos críticos

In [10]:
criticos = ['id_transaccion', 'id_producto', 'cantidad', 'fecha', 'nombre_producto', 'precio_unitario', 'total_venta']
print(ventas[criticos].isna().sum())
assert ventas[criticos].isna().sum().sum() == 0
print('\nFilas finales:', len(ventas))
ventas.dtypes

id_transaccion     0
id_producto        0
cantidad           0
fecha              0
nombre_producto    0
precio_unitario    0
total_venta        0
dtype: int64

Filas finales: 289


id_transaccion              int64
id_producto                 int64
cantidad                    int64
fecha              datetime64[us]
nombre_producto               str
precio_unitario           float64
total_venta               float64
dtype: object

In [11]:
ventas.groupby('nombre_producto')['total_venta'].sum().sort_values(ascending=False).round(2)

nombre_producto
Queso dambo kg         73840.0
Café molido 500g       41000.0
Aceite girasol 1.5L    37194.3
Yerba mate 1kg         32368.0
Dulce de leche 450g    19060.5
Pan de molde           10925.0
Fideos 500g             7326.0
Arroz 1kg               6873.0
Galletitas maría        4402.0
Leche entera 1L         4368.0
Name: total_venta, dtype: float64

## 5. Exportación a Parquet

In [12]:
OUT.parent.mkdir(parents=True, exist_ok=True)
ventas.to_parquet(OUT, index=False, engine='pyarrow')

# Verificación: se relee el archivo y se comparan tamaños y tipos
check = pd.read_parquet(OUT)
print(check.shape)
print(check.dtypes)
print(f"Parquet generado: {OUT} ({OUT.stat().st_size/1024:.1f} KB)")

(289, 7)
id_transaccion              int64
id_producto                 int64
cantidad                    int64
fecha              datetime64[us]
nombre_producto               str
precio_unitario           float64
total_venta               float64
dtype: object
Parquet generado: ../data/processed/ventas_final.parquet (8.9 KB)


## Conclusiones
- Se integraron dos fuentes de distinto formato (CSV y Excel) usando `id_producto` como llave, unificando previamente su tipo (`int64`).
- Las fechas quedaron como `datetime64` y se eliminaron registros no atribuibles (sin producto, sin fecha o con producto inexistente).
- Las cantidades faltantes se imputaron con la mediana por producto.
- El dataset final no tiene nulos en campos críticos, incluye `total_venta` y se guardó en Parquet: formato columnar y comprimido que preserva los tipos (incluida la fecha) y escala mejor que CSV en volúmenes grandes.